This part of the pipeline dereplicates the set of clostridial genomes at NCBI by ANI.

### Checking dependencies

In [ ]:
conda activate skder
skder -v
conda deactivate

### Paths and parameters

#### Pipeline input folders

In [ ]:
downloads="/mnt/STORAGE/downloads/clostridia/genomes/"

#### Pipeline output folders

In [ ]:
task_root="./00-dereplication"
QCed_data="$task_root/data_QC1"
skder_test="$task_root/skder_test"
dereplication="$task_root/skder"

mkdir -p $task_root $QCed_data $skder_test $dereplication

#### Tool pointers and parameters

In [ ]:
n_cores=22
ani=96
af=50

### Downloading raw metadata table

Go to NCBI Datasets online and search for "Clostridia".

Apply the following filters:
- Annotated in RefSeq
- Exclude MAGs
- Exclude atypicals
  
Select all hits, download the resulting table including the following columns as a tsv (`genomes_metadata`), import it into Excel and save as a spreadsheet (`genomes_metadata.xlsx`).
- Assembly
- GenBank
- RefSeq
- Scientific Name
- Tax ID
- Modifier
- Size (Mb)
- Contigs
- Level
- Release date
- Contig N50 (kb)
- GC percent
- Genes
- CheckM completeness (%)
- CheckM contamination (%)
  
Apply a filtering criterion in Excel to do the QC using the following filters:
- N50 >= 50000
- Contigs <= 200
- Completeness >= 90
- Contamination < 10

Copy-paste the passed assembly IDs into a new file `passed_QC` in this task's root folder.

Make soft links to the original downloaded sequence data for the passed IDs.

In [ ]:
cat $task_root/passed_QC | xargs -I % ln -s -t $QCed_data $downloads/%.fna.gz

### Do a small sensitivity analysis of the dereplication parameters

In [ ]:
conda activate skder
skder -g $QCed_data -o $skder_test -tc -c $n_cores
conda deactivate

The usual species ANI threshold of 96% seems appropriate combined with an AF threshold of 50%.

### Do the dereplication

In [ ]:
conda activate skder
skder -g $QCed_data -o $dereplication -c $n_cores -i $ani -a $af -n -l
conda deactivate

### Filter the metadata table after dereplication

In [ ]:
join -j 1 <(sort $task_root/dereplicated) <(sort genomes_metadata) -t $'\t' | sort -k 2 > $task_root/_dereplicated_metadata
cat <(head -1 genomes_metadata) $task_root/_dereplicated_metadata > $task_root/dereplicated_metadata
rm $task_root/_dereplicated_metadata
cp $task_root/dereplicated_metadata ./

### Calculate the size of the dereplication clusters

In [ ]:
python utils/get_dereplication_cluster_metrics.py $dereplication/skDER_Clustering.txt
mv cluster_metrics $task_root/